# CI Digital — figuras geométricas do Python ao SystemVerilog

**Versão 2.0 histórica | Fluxo original 70/15/15, sem FPGA.**
A campanha atual 70/20/10 e cinco folds usa `ci_digital_cv_train.py`, os manifestos em `runs/cv_70_20_10/` e a validação em `results/rtl_cv_70_20_10/`. Este notebook preserva a execução histórica.

Este notebook usa os módulos da pasta `python/` do **ZIP completo**. Não é um notebook isolado.
O dataset já contém **2.000 PNGs 28×28**, divididos em treino, validação e teste.
Há uma execução de referência em `runs/baseline/` e pesos já exportados em `exports/trained/`.

**Status atual:** a compilação e a simulação SystemVerilog foram executadas com Icarus. Os seis modelos da campanha atual coincidem com o Python inteiro nas mesmas 200 imagens de teste, conforme `results/rtl_cv_70_20_10/summary.json`. Os testes Python, isoladamente, não provam execução HDL.

Siga as seções em ordem. Para estudar o RTL imediatamente, use `eda_playground/trained/`;
não é necessário aguardar um novo treinamento.

## 0. Somente para uso no Colab, opcional

No computador, pule esta célula. No Colab, altere `UPLOAD_ZIP_NO_COLAB` para `True` e envie o ZIP
completo. Não é necessário conectar o Google Drive. O ambiente do Colab é separado da sua máquina;
guarde os resultados antes de encerrar a sessão.

In [ ]:
UPLOAD_ZIP_NO_COLAB = False
if UPLOAD_ZIP_NO_COLAB:
    from google.colab import files
    import zipfile
    from pathlib import Path
    enviados = files.upload()
    destino = Path('/content').resolve()
    for nome in enviados:
        if not nome.lower().endswith('.zip'):
            continue
        with zipfile.ZipFile(nome) as z:
            for membro in z.infolist():
                alvo = (destino / membro.filename).resolve()
                if not alvo.is_relative_to(destino):
                    raise ValueError('Caminho inseguro no ZIP')
            z.extractall(destino)

## 1. Encontrar o projeto e conferir o ambiente

No Windows, use o ambiente virtual do guia Python. Uma GPU é opcional. A arquitetura é pequena
para permitir também treinamento em CPU. Não reinstale um PyTorch com CUDA que já funciona sem necessidade.

In [ ]:
from pathlib import Path
import json, sys, inspect, zipfile
import numpy as np
import torch
from IPython.display import display, Image, Markdown

candidatos = [Path.cwd(), Path.cwd().parent, Path.cwd()/'ci_digital_v2', Path('/content/ci_digital_v2')]
ROOT = next((p.resolve() for p in candidatos if (p/'rtl'/'tiny_cnn.sv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Extraia o ZIP completo e abra o notebook dentro da pasta notebooks/.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from python.shapes import (CLASSES, SEED, ShapeCNN, load_split, train_model,
                           generate_dataset, shape_image)
from python.quantization import run_export, integer_forward
from scripts.make_eda import make_eda

print('Pasta:', ROOT)
print('PyTorch:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
print('Classes:', dict(enumerate(CLASSES)))

## 2. Examinar as imagens e a separação

Cada classe tem 280 imagens de treino, 60 de validação e 60 de teste. Há rotação, translação,
escala, variação de intensidade, ruído e borrão leves. As figuras são geradas independentemente
por partição; não dividimos versões aumentadas da mesma imagem-base entre treino e teste.

A ordem das classes é fixa: círculo, quadrado, triângulo, cruz e estrela.
Os nomes de pasta não devem ser remapeados por ordem alfabética.

In [ ]:
DATASET = ROOT/'dataset'
info = json.loads((DATASET/'dataset_info.json').read_text(encoding='utf-8'))
print(json.dumps(info, indent=2, ensure_ascii=False))
display(Image(filename=str(DATASET/'preview.png')))

## 3. Gerar outro dataset, opcional

O conjunto solicitado já está pronto. Este bloco fica desativado para não substituir seus dados.
Novas imagens não garantem, sozinhas, melhor acurácia. Preserve um teste fixo quando comparar modelos.
A implementação completa do gerador está em `python/shapes.py`, função `shape_image`.

In [ ]:
GERAR_NOVO_DATASET = False
if GERAR_NOVO_DATASET:
    NOVO_DATASET = ROOT/'dataset_extra'
    generate_dataset(NOVO_DATASET, total=2000, seed=SEED+100, overwrite=False)
    print('Novo conjunto criado em', NOVO_DATASET)
# Não mudamos DATASET automaticamente: mudar o teste altera a comparação com o baseline.

## 4. Entender a arquitetura

`28×28×1 → Conv(8,3×3) → ReLU → Pool2×2 → Conv(16,3×3) → ReLU → Pool2×2 → Flatten400 → Dense5`

São **3.253 parâmetros**. As convoluções não têm padding; os tamanhos são 28→26→13→11→5.
O último pooling descarta a borda ímpar, exatamente como o RTL. Não há softmax no núcleo de inferência;
a classe vem do maior logit. O treinamento usa `CrossEntropyLoss` diretamente nos logits.

**Não altere os canais, os tamanhos ou a ordem do flatten sem alterar também o RTL.**

In [ ]:
modelo_exemplo = ShapeCNN()
print(modelo_exemplo)
print('Parâmetros:', sum(p.numel() for p in modelo_exemplo.parameters()))
print(inspect.getsource(ShapeCNN))

## 5. Configurar o treinamento

Por padrão, as próximas células fazem um novo treino. Para apenas explorar os pesos já fornecidos,
altere `TREINAR_AGORA=False`. A melhor época é selecionada por acurácia de validação, com desempate
pela perda de validação. O teste só é avaliado depois dessa seleção.

Uma execução existente não é sobrescrita sem `SOBRESCREVER=True`. Trocar os nomes das pastas é a forma
mais segura de preservar experimentos.

In [ ]:
TREINAR_AGORA = True
EPOCHS = 40
BATCH = 64
LR = 0.002
DEVICE = 'auto'  # ou 'cpu'; 'cuda' exige uma instalação CUDA compatível
SOBRESCREVER = False
RUN_DIR = ROOT/'runs'/'meu_treino'
EXPORT_DIR = ROOT/'exports'/'meu_treino'
EDA_DIR = ROOT/'eda_playground'/'meu_treino'
if not TREINAR_AGORA:
    RUN_DIR = ROOT/'runs'/'baseline'
    EXPORT_DIR = ROOT/'exports'/'trained'
    EDA_DIR = ROOT/'eda_playground'/'trained'
print(RUN_DIR, EXPORT_DIR, sep='\n')

## 6. Treinar ou abrir a execução de referência

A função salva `best.pt`, `history.csv`, relatório JSON e previsões do teste. Não existe treinamento
no SystemVerilog: somente a inferência recebe os parâmetros aprendidos aqui.

A entrada é **pixel/256**, e não pixel/255. Essa escolha facilita representar a escala por potência de dois.
Use exatamente a mesma convenção durante todo o pipeline.

In [ ]:
if TREINAR_AGORA:
    relatorio_treino = train_model(
        dataset=DATASET, out=RUN_DIR, epochs=EPOCHS, batch=BATCH, lr=LR,
        seed=SEED, device=DEVICE, patience=10, overwrite=SOBRESCREVER)
else:
    relatorio_treino = json.loads((RUN_DIR/'training_report.json').read_text(encoding='utf-8'))
print('Melhor época:', relatorio_treino['best_epoch'])
print('Teste float:', relatorio_treino['float_test'])

## 7. Visualizar a evolução

Acompanhe treino e validação. Não escolha a época pela curva de teste. Seeds ajudam na repetibilidade,
mas não garantem resultados idênticos entre diferentes bibliotecas, dispositivos e versões.

In [ ]:
import csv
import matplotlib.pyplot as plt
with (RUN_DIR/'history.csv').open(encoding='utf-8') as f:
    historico = list(csv.DictReader(f))
ep = [int(r['epoch']) for r in historico]
plt.figure(figsize=(8,4))
plt.plot(ep, [float(r['train_accuracy']) for r in historico], label='Treino')
plt.plot(ep, [float(r['val_accuracy']) for r in historico], label='Validação')
plt.xlabel('Época'); plt.ylabel('Acurácia'); plt.ylim(0,1.02)
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(ep, [float(r['train_loss']) for r in historico], label='Treino')
plt.plot(ep, [float(r['val_loss']) for r in historico], label='Validação')
plt.xlabel('Época'); plt.ylabel('Perda'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 8. Quantizar e exportar o contrato completo

Pixels e ativações são `uint8`; pesos são `int8`; biases e acumuladores são `int32`.
As escalas são potências de dois por camada. A calibração usa **500 imagens do treino**, nunca do teste.

O exportador salva pesos, biases, shifts, metadados, imagens e todas as respostas intermediárias.
O arredondamento das ativações é explícito: ReLU → arredondar metade para cima → shift → saturar 0…255.
O modelo inteiro pode discordar do float em algumas imagens. A referência exigida para o RTL é o **inteiro**.

In [ ]:
if TREINAR_AGORA:
    avaliacao = run_export(RUN_DIR/'best.pt', EXPORT_DIR, DATASET,
                          per_class=3, random_demo=False, overwrite=SOBRESCREVER)
else:
    avaliacao = json.loads((EXPORT_DIR/'evaluation.json').read_text(encoding='utf-8'))
print('Acurácia float:', avaliacao['float_test']['accuracy'])
print('Acurácia inteira:', avaliacao['integer_test']['accuracy'])
print('Concordância de classe float/inteiro:', avaliacao['float_int_class_agreement'])
print('Matriz de confusão inteira; linhas=reais, colunas=previstas')
print(CLASSES)
print(np.asarray(avaliacao['integer_test']['confusion_matrix']))
quant = json.loads((EXPORT_DIR/'quantization.json').read_text(encoding='utf-8'))
for camada in quant['layers']:
    print(camada)

## 9. Examinar a referência inteira de uma imagem

Estes cinco logits são os números que o SystemVerilog precisa reproduzir. Não são probabilidades.
O exemplo usa uma imagem do teste e não uma posição arbitrária do dataset completo.

In [ ]:
q = dict(np.load(EXPORT_DIR/'quantized_model.npz'))
xt, yt, caminhos = load_split(DATASET, 'test')
INDICE = 0
traces = integer_forward(xt[INDICE:INDICE+1], q, trace=True)
print('Imagem:', caminhos[INDICE], '| real:', CLASSES[int(yt[INDICE])])
print('Logits inteiros:', traces['logits'][0])
print('Predição:', CLASSES[int(traces['logits'][0].argmax())])
for nome, tensor in traces.items():
    print(nome, tensor.shape, tensor.dtype)

## 10. Preparar o EDA Playground

O gerador junta os quatro módulos RTL e embute parâmetros + cinco imagens (uma de cada classe)
+ resultados esperados no testbench. Copie os arquivos da **mesma pasta** para os dois painéis do site.

Isso **prepara**, mas não executa, a simulação. O teste do navegador compara todos os mapas,
os logits e as classes. Consulte o PDF Verilog para selecionar o simulador e interpretar falhas.

In [ ]:
pacote_eda = make_eda(EXPORT_DIR, EDA_DIR, samples=5)
print('Painel Design:', EDA_DIR/'design.sv')
print('Painel Testbench:', EDA_DIR/'testbench.sv')
print('Depois da execução bem-sucedida, o console deverá terminar com PASS_ALL.')

## 11. Testes locais, opcionais

O primeiro comando verifica o pipeline Python. O segundo exige os executáveis `iverilog` e `vvp` no PATH.
O script do simulador informa erro quando eles não estão disponíveis; não inventa um resultado de simulação.

In [ ]:
# Execute em um terminal aberto na raiz do projeto:
# python -m unittest discover -s tests -v
# python scripts/run_sim.py --export exports/meu_treino
# Para ondas, acrescente --waves; o VCD pode ficar grande.
# No modo baseline use --export exports/trained.

## 12. Guardar os resultados para a próxima etapa

O ZIP abaixo contém o checkpoint, métricas, exportação inteira e versão online. Ele é menor do que o projeto
completo porque não copia os 2.000 PNGs novamente. Guarde também os logs do simulador quando rodá-lo.
Não conclua que o hardware está correto apenas porque acertou uma classe: compare os logits e os mapas.

In [ ]:
saida = ROOT/'resultados_para_verilog.zip'
with zipfile.ZipFile(saida, 'w', zipfile.ZIP_DEFLATED) as z:
    for pasta, prefixo in [(RUN_DIR,'run'), (EXPORT_DIR,'export'), (EDA_DIR,'eda_playground')]:
        for arquivo in sorted(pasta.rglob('*')):
            if arquivo.is_file():
                z.write(arquivo, str(Path(prefixo)/arquivo.relative_to(pasta)))
print('Resultados salvos em:', saida)
# Somente no Colab, para salvar no seu computador:
# from google.colab import files
# files.download(str(saida))

## Limites do experimento

Acurácia em figuras sintéticas não mede desempenho em imagens reais de drones. Este é um classificador
fechado em cinco classes: ele não detecta posição, não reconhece fundo como uma sexta classe e não comanda pouso.
Não há síntese, medição de frequência, consumo ou validação na DE10-Lite neste escopo.

### Referências oficiais
- [PyTorch Conv2d](https://docs.pytorch.org/docs/2.10/generated/torch.nn.Conv2d.html)
- [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/2.10/generated/torch.nn.CrossEntropyLoss.html)
- [PyTorch Reproducibility](https://docs.pytorch.org/docs/2.10/notes/randomness.html)
- [EDA Playground — Settings](https://eda-playground.readthedocs.io/en/latest/settings.html)
- [Icarus Verilog — Getting Started](https://steveicarus.github.io/iverilog/usage/getting_started.html)